# Astrometry Notebook (version 2)

In order to properly work with our data we need to know what part of the sky each image is of. Figuring this out is called *plate solving*; the easiest way for us to do so is to uplaod each of our images to nova.astronomy.net. 
Doing this by hand is  possible but tedious, so this notebook helps do it automatically.

The general process is fairly simple:
1. Use our API key to let the server know we're allowed to send it a bunch of things and that it can trust us.
2. Make a list of every file that we've placed in a specific 'input' folder.
3. Upload each of those files to Astrometry and get the solved header world coordinate system (wcs) information.
4. Add the new wcs information to the fits image header and save the fits image (now with added astrometry) to the 'output' folder.

**Note that this is modified version of the original AstrometryNotebook because the previous version wasn't able to download the solved images from the nova.astronomy.net website.**

To run with notebook you will need to have installed both the astropy and astroquery packages. If you use pip you could do this by running in a terminal:

pip install astropy

pip install astroquery




# Import Statements and Required Functions

In [1]:
import sys
#My QAOP folder is in: /Users/aidanmcclung/Desktop/Summer_Exoplanets

codeFilePath = '/Users/fissel/git/418QAOP/'  #/QAOP/ <- folder containing QAOP folder.
sys.path.append(codeFilePath) #this lets python know to look here for import statements

In [2]:
from astroquery.astrometry_net import AstrometryNet
from astropy.io import fits

In [3]:
import QAOP.client as client #This is a .py file that should be in the QAOP folder

import os #OS is essentially just the basic terminal comand line interface tools in python. 
#We need it to move our files around.

#We need two packages in order to do our web operations:
import requests #The default python webserver interfacing package
import json #The python JavaScript Object Notation interfacing package

from IPython.display import clear_output #to save you from the worlds longest output cells :)


from QAOP.QAOP_utils import readConfigFile
extraCodeFilePath, dataFilePath, errormsg = readConfigFile( configPath= codeFilePath + 'QAOP/')
print(errormsg) 
#print(dataFilePath)#you can check this looks right

no errors reading config file


## Function Definitions

I created each of these functions, and have attempted to add enough comments to describe what they do.
Knowing those specifics isn't necessary though, and if you're not interested you can absolutely **run each of these cells once** and move on to the **Main Program**

#### (You can run each of these cells and move on to the "Main Program")

In [11]:
def findFileName(fn,debug=False):
    ''' This function scans the end of a filename to see whether its a .fit or .fits, 
    and then takes the corresponding 3 numbers that it needs for the file name. 
    This is important because where those three characters we need are will change between the two file types.'''
    if fn[-4:] == ".fit": 
        name = fn[:-4]
        if debug: print(".fit file detected.  Saving filename as:",name)
    if fn[-5:] == ".fits": 
        name = fn[:-5]
        if debug: print(".fits file detected. Saving filename as:",name)
    return name


In [12]:
def submitInpFile(fn,debug=False):
    """This function is the wrapper for uploading files; it uses the above getSubID to upload the image, 
    but it also finds and saves the corresponding name alongside the subID, so that we can get them later."""
    name = findFileName(fn) #detect whether .fits or .fit so we can get the right characters for the name
    if debug: print("Submitting File:",name)
    # Here is where we send the request to solve the image
    wcs_header = ast.solve_from_image(dataFilePath+'input/'+fn)
    return name,wcs_header #We return the name and the solved header

def saveOutputFile(name,wcs_header):
    fileinname = (dataFilePath+'input/'+name+'.fit')
    fileoutname = (dataFilePath+'output/'+name+'.fits')
    hdul = fits.open(fileinname)
    header = hdul[0].header
    header.update(wcs_header)
    hdul.writeto(fileoutname,overwrite=True)
    


# Main Program

The first step is to enter an api key. Publically releasing API keys to the internet is a bad idea, so this key should be something you can find on the onQ page. Track it down, and copy it in order to paste it into the textbox that appears when you run the following cell.

We are also going to set up our interface with astrometry.net

In [22]:
ast = AstrometryNet()
# CHANGE THIS TO ADD YOUR API KEY
ast.api_key = 'xxxxxxxxxxxxxxx'


-----
Now that we're logged in, we next need to assemble a list of all the files we'd like to upload. 

These should have been put into the 'input' folder with the renaming section in the master notebook, so they should have already been filtered and only be images, but we will double check because sometimes there's hidden files that have been added by programs like jupyter.

In [7]:
inputFiles_all = os.listdir(dataFilePath+"input")
#print(inputFiles_all)
#The program will pull in ANY files that are there; 
#sometimes there are hidden os files or such we don't want, so we need to filter those out.
inputFiles = []
for file in inputFiles_all:
    if file[-4:] == '.fit': inputFiles.append(file)
#print(inputFiles)

numFiles = len(inputFiles)
print(numFiles,'files were identified:\n',inputFiles)

3 files were identified:
 ['calibrated-T11-group4b_2026-HAT-P-56-20261025-014459-V-BIN1-E-090-003.fit', 'calibrated-T11-group4b_2026-HAT-P-56-20261025-014314-V-BIN1-E-090-002.fit', 'calibrated-T11-group4b_2026-HAT-P-56-20261025-014106-V-BIN1-E-090-001.fit']


Here we set up some counters to keep track of how many images have been solved.

If you want to rerun your astrometry solutions you will also need to rerun this cell.

In [15]:
#Set these two here in a seperate cell to maintain progress upon hitting an error. 
# I wasn't fully able to test that it resumed under all circumstances, so I would advise not fully counting on it.
upload_progress = 0 
count_success = 0
count_failed = 0
checkFiles = []

Here is the code where we cycle through each of our images and:
1. upload our image data to astrometry (submitInpFile)
2. download the world coordinate system solutions, add them to the input image header file, and write the fits image with astrometry data to the same filename in the "output" folder (but with the file ending '.fits' instead of '.fit')

**Note: it will take quite some time to upload and solve all of your images.**

In [16]:
while upload_progress < numFiles:
    file = inputFiles[upload_progress]
    
    #Take care of some of the feedback stuff so we know how it's doing
    print("")
    print('Upload Status:',upload_progress,'of',numFiles, "   Current File (Unordered):",file)
    print("")
    name,wcs_header = submitInpFile(file)
    # Retrieve the fits header which we will update with the astrometry.net WCS solution
    if wcs_header:
        count_success += 1
        saveOutputFile(name,wcs_header)
    else:
        print("Warning failed to get a WCS solution for image",name)
        count_failed += 1
    upload_progress += 1
print("")    
print("Solving Complete. solved ",count_success," failed",count_failed,"out of ",numFiles,"images")


Upload Status: 0 of 3    Current File (Unordered): calibrated-T11-group4b_2026-HAT-P-56-20261025-014459-V-BIN1-E-090-003.fit

Solving.............Upload Status: 1 of 3    Current File (Unordered): calibrated-T11-group4b_2026-HAT-P-56-20261025-014314-V-BIN1-E-090-002.fit

Solving.............Upload Status: 2 of 3    Current File (Unordered): calibrated-T11-group4b_2026-HAT-P-56-20261025-014106-V-BIN1-E-090-001.fit

Solving............
Solving Complete. solved  3  failed 0 out of  3 images


#### We've now done everything that we need to with this notebook, and you can go back to the Master and continue.

# Before/After Comparison (Optional)

There are a few more things that may be interesting to you that we wanted to include for you to check out.

If you're interested in seeing what changed by doing this process, you can do so with the cells that follow. There are two versions, one which demonstrates the WCS info we added, and another one below that which prints the whole header for you to see.

In [18]:
#These two packages are only used to check on whether the program worked or not.
from astropy.io import fits #be able to interpret fits files
from astropy.wcs import WCS #convert/extract the information we added

In [19]:
inputFileName = "calibrated-T11-group4b_2026-HAT-P-56-20261025-014106-V-BIN1-E-090-001.fit"

print("WCS Info Before Astrometry:")
print(" ")

with fits.open(dataFilePath+"input/"+inputFileName) as f:
    w = WCS(f[0].header)
    print(w)

print(" ")
print("----------------------------------------------------------------")
print("WCS Info After Astrometry:")
print(" ")

outputFileName = inputFileName + "s" #change this if the only change wasn't .fit -> .fits
with fits.open(dataFilePath+"output/"+outputFileName, ignore_missing_simple=True) as f:
    w = WCS(f[0].header)
    print(w)

WCS Info Before Astrometry:
 
WCS Keywords

Number of WCS axes: 2
CTYPE : '' '' 
CRVAL : 0.0 0.0 
CRPIX : 0.0 0.0 
PC1_1 PC1_2  : 1.0 0.0 
PC2_1 PC2_2  : 0.0 1.0 
CDELT : 1.0 1.0 
NAXIS : 4008  2672
 
----------------------------------------------------------------
WCS Info After Astrometry:
 
WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN-SIP' 'DEC--TAN-SIP' 
CRVAL : 100.639981275 27.3587021096 
CRPIX : 1197.08107758 1806.8253479 
CD1_1 CD1_2  : 0.000227391978535 -1.71602782648e-06 
CD2_1 CD2_2  : 1.64523922364e-06 0.000227291386263 
NAXIS : 4008  2672


the RADECSYS keyword is deprecated, use RADESYSa. [astropy.wcs.wcs]


In [21]:
inputFileName = "calibrated-T11-group4b_2026-HAT-P-56-20261025-014106-V-BIN1-E-090-001.fit"
 #Change this to be what file you'd like to check

print("FITS Header Before Astrometry:")
print(" ")

with fits.open(dataFilePath+"input/"+inputFileName) as f:
    print(repr(f[0].header))

print(" ")
print("----------------------------------------------------------------")
print("FITS Header After Astrometry:")
print(" ")

outputFileName = inputFileName + "s" #change this if the only change wasn't .fit -> .fits
with fits.open(dataFilePath+"output/"+outputFileName) as f:
    print(repr(f[0].header))

FITS Header Before Astrometry:
 
SIMPLE  =                    T                                                  
BITPIX  =                  -32 /8 unsigned int, 16 & 32 int, -32 & -64 real     
NAXIS   =                    2 /number of axes                                  
NAXIS1  =                 4008 /fastest changing axis                           
NAXIS2  =                 2672 /next to fastest changing axis                   
BSCALE  =   1.0000000000000000 /physical = BZERO + BSCALE*array_value           
BZERO   =  0.00000000000000000 /physical = BZERO + BSCALE*array_value           
DATE-OBS= '2026-10-26T07:41:33.130' / [ISO 8601] UTC date/time of exposure start
EXPTIME =   9.00000000000E+001 / [sec] Duration of exposure                     
EXPOSURE=   9.00000000000E+001 / [sec] Duration of exposure                     
SET-TEMP=  -25.000000000000000 /CCD temperature setpoint in C                   
CCD-TEMP=  -25.187500000000000 /CCD temperature at start of exposure in C   